# 🤖 AIOS Quant ML Training

Обучение предсказательных моделей на биржевых данных (XGBoost, LightGBM, CatBoost, LSTM, Transformer).

**Среда выполнения → Сменить тип → T4 GPU** (для LSTM/Transformer).

### Источники данных (2 способа)
- **(а)** Прямая загрузка свечей через `ccxt` (Binance/Bybit/OKX/Kraken) прямо в Colab.
- **(б)** Загрузите `latest.tar.gz` с VPS (папка `data/quant/export/`) в сессию Colab.

Модели сохраняются и публикуются в **Hugging Face Hub** (опционально, введите токен) или качаются VPS.

In [ ]:
# === ЯЧЕЙКА 1: Установка зависимостей ===
!pip install -q ccxt pandas numpy scikit-learn xgboost lightgbm catboost torch
import ccxt, pandas as pd, numpy as np
print('✅ Зависимости установлены, ccxt', ccxt.__version__)

In [ ]:
# === ЯЧЕЙКА 2: Загрузка данных ===
import os, io, tarfile, pandas as pd

def load_data():
    rows = []
    # (б) если загружен датасет с VPS
    if os.path.exists('latest.tar.gz'):
        tar = tarfile.open('latest.tar.gz', 'r:gz')
        for m in tar.getmembers():
            if m.isfile() and m.name.endswith('_1h.csv'):
                df = pd.read_csv(tar.extractfile(m))
                df['symbol'] = m.name.split('/')[0]
                rows.append(df)
        tar.close()
    if rows:
        return pd.concat(rows, ignore_index=True)
    # (а) прямая загрузка через ccxt
    clients = {'binance': ccxt.binance(), 'bybit': ccxt.bybit(), 'okx': ccxt.okx()}
    symbols = ['BTC/USDT','ETH/USDT','SOL/USDT','BNB/USDT']
    for name, cl in clients.items():
        cl.load_markets()
        for s in symbols:
            try:
                o = cl.fetch_ohlcv(s, '1h', limit=500)
                df = pd.DataFrame(o, columns=['ts','open','high','low','close','volume'])
                df['symbol'] = s.replace('/','_')
                df['exchange'] = name
                rows.append(df)
            except Exception as e:
                print('skip', name, s, e)
    return pd.concat(rows, ignore_index=True)

df = load_data()
df['ts'] = pd.to_datetime(df['ts_ms'] if 'ts_ms' in df else df['ts'], unit='ms', errors='coerce')
print('✅ Данные:', df.shape)
print(df.head(2).to_string())

In [ ]:
# === ЯЧЕЙКА 3: Признаки (feature engineering) ===
from sklearn.model_selection import TimeSeriesSplit

def make_features(g):
    g = g.sort_values('ts')
    g['ret1'] = g['close'].pct_change()
    g['ema12'] = g['close'].ewm(span=12).mean()
    g['ema26'] = g['close'].ewm(span=26).mean()
    g['rsi'] = 100 - 100/(1 + g['close'].pct_change().rolling(14).mean()/
                        g['close'].pct_change().rolling(14).std().replace(0,1e-9))
    g['vol_ma'] = g['volume'].rolling(20).mean()
    g['target'] = (g['close'].shift(-1) > g['close']).astype(int)  # движение вверх через 1 бар
    return g

df = df.groupby('symbol').apply(make_features).reset_index(drop=True)
df = df.dropna().reset_index(drop=True)
features = ['open','high','low','close','volume','ret1','ema12','ema26','rsi','vol_ma']
X = df[features].values; y = df['target'].values
print('✅ Признаки готовы:', X.shape)

In [ ]:
# === ЯЧЕЙКА 4: Обучение (XGBoost, LightGBM, CatBoost) ===
import xgboost as xgb, lightgbm as lgb, catboost as cb
from sklearn.metrics import accuracy_score
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=3)
results = {}
for name, model in [
    ('XGBoost', xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.05, use_label_encoder=False, eval_metric='logloss')),
    ('LightGBM', lgb.LGBMClassifier(n_estimators=200, max_depth=6, learning_rate=0.05, verbose=-1)),
    ('CatBoost', cb.CatBoostClassifier(iterations=200, depth=6, learning_rate=0.05, verbose=0)),
]:
    accs = []
    for tr, te in tscv.split(X):
        m = model.__class__(**model.get_params()) if hasattr(model,'get_params') else model
        m.fit(X[tr], y[tr])
        accs.append(accuracy_score(y[te], m.predict(X[te])))
    results[name] = np.mean(accs)
    print(f'  {name}: {np.mean(accs):.4f}')
print('\n✅ Лучшая табличная модель:', max(results, key=results.get))

In [ ]:
# === ЯЧЕЙКА 5: Обучение LSTM (PyTorch) ===
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

SEQ = 20
def seq_xy(X_, y_):
    xs, ys = [], []
    for i in range(SEQ, len(X_)):
        xs.append(X_[i-SEQ:i]); ys.append(y_[i])
    return np.array(xs, dtype=np.float32), np.array(ys, dtype=np.float32)
Xs, ys = seq_xy(X, y)
cut = int(len(Xs)*0.8)
tr_x, te_x = Xs[:cut], Xs[cut:]
tr_y, te_y = ys[:cut], ys[cut:]

class LSTMModel(nn.Module):
    def __init__(self, nfeat, hidden=32):
        super().__init__()
        self.lstm = nn.LSTM(nfeat, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, 2)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = LSTMModel(Xs.shape[2]).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
lossf = nn.CrossEntropyLoss()
tra = TensorDataset(torch.tensor(tr_x), torch.tensor(tr_y, dtype=torch.long))
tel = TensorDataset(torch.tensor(te_x), torch.tensor(te_y, dtype=torch.long))
tld = DataLoader(tra, batch_size=64, shuffle=True)
eld = DataLoader(tel, batch_size=128)
model.train()
for ep in range(15):
    tl = 0
    for xb, yb in tld:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad(); loss = lossf(model(xb), yb); loss.backward(); opt.step()
        tl += loss.item()
    # val acc
    model.eval(); ok=0; tot=0
    with torch.no_grad():
        for xb, yb in eld:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb).argmax(1); ok+=(pred==yb).sum().item(); tot+=yb.size(0)
    model.train()
    print(f'  epoch {ep+1}: loss={tl/len(tld):.4f} val_acc={ok/tot:.4f}')
print('✅ LSTM обучена')

In [ ]:
# === ЯЧЕЙКА 6: Сохранение моделей ===
import joblib
os.makedirs('models', exist_ok=True)
# Сохраняем лучшую табличную модель (пример - CatBoost)
best = cb.CatBoostClassifier(iterations=200, depth=6, learning_rate=0.05, verbose=0)
best.fit(X, y)
best.save_model('models/catboost_price_dir.cbm')
joblib.dump(best, 'models/catboost_price_dir.pkl')
print('✅ Модели сохранены в /content/models')
print('   Загрузите их на VPS или в HF Hub для инференса.')